TWITTER PRODUCT SENTIMENT ANALYSIS

AUTHOR : Charity Nduati

1. BUSINESS UNDERSTANDING

1.1. Business Context

This project analyzes Twitter posts from the SXSW Interactive Festival, where Apple and Google were heavily promoting their products and services. The goal is to use sentiment analysis to monitor public reactions in real time and help organizations quickly identify customer concerns, brand perception, and emerging public relations risks.

1.2. Business Problem

A large proportion of tweets are neutral and provide little actionable information. Manually reviewing thousands of tweets is inefficient and can delay responses to important customer complaints, allowing negative issues to spread rapidly online.

1.3. Stakeholders

Corporate Communications & Brand PR Teams: Need visibility into overall brand sentiment and early detection of potential PR issues.


Customer Experience (CX) Teams: Need a system that automatically identifies and prioritizes customer complaints and technical issues for faster response.

1.4. Business Questions

How can natural language processing techniques be used to automatically filter neutral tweets and focus attention on emotionally charged customer feedback?


Can a machine learning model accurately identify negative sentiment, particularly the minority class of negative tweets, to ensure critical customer issues are detected and addressed quickly?

1.5. Project Objectives

1.5.1. Primary Objective

Develop a sentiment classification model that automatically categorizes tweets as positive, neutral, or negative.

1.5.2. Specific Objectives

Reduce information overload by filtering non-actionable neutral tweets.

Identify negative customer experiences and potential PR risks as early as possible.

Improve response times for customer support teams.

Evaluate machine learning models and optimize performance, particularly for detecting negative sentiment.

Use TF-IDF vectorization to extract meaningful information from tweet text.

2. DATA UNDERSTANDING

2.1.Data Source

The dataset was obtained from CrowdFlower via data.world and contains Twitter posts related to Apple and Google products during SXSW.

2.2. Dimensions

The dataset contains $9,093$ rows and $3$ columns.

2.3. Column Names & Data Types

Tweet_text (Object/String): The raw text of the tweet ($9,092$ non-null values, $1$ missing value).

Emotion_in_tweet_is_directed_at (Object/String): The specific product or brand being targeted ($3,291$ non-null values, high missingness).

Is_there_an_emotion_directed_at_a_brand_or_product (Object/String): Our target label ($9,093$ non-null values).

2.4. Class Distribution of the Target

No emotion toward brand or product: $5,389$ tweets

TweetsPositive emotion: $2,978$ tweets

TweetsNegative emotion: $570$ tweets

TweetsI can't tell: $156$ tweets

CODE ENVIRONMENT SET UP

In [42]:
#importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import regexp_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Download NLTK dependencies safely
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Plotting aesthetics
sns.set_theme(style="whitegrid")
%matplotlib inline

3. DATA LOADING, INSPECTION  AND CLEANING

3.1. Data Loading & Inspection

In [26]:
#Data loading
raw_df = pd.read_csv('judge-1377884607_tweet_product_company.csv', encoding='latin1')

In [27]:
raw_df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [28]:
raw_df.info

<bound method DataFrame.info of                                              tweet_text  \
0     .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1     @jessedee Know about @fludapp ? Awesome iPad/i...   
2     @swonderlin Can not wait for #iPad 2 also. The...   
3     @sxsw I hope this year's festival isn't as cra...   
4     @sxtxstate great stuff on Fri #SXSW: Marissa M...   
...                                                 ...   
9088                      Ipad everywhere. #SXSW {link}   
9089  Wave, buzz... RT @mention We interrupt your re...   
9090  Google's Zeiger, a physician never reported po...   
9091  Some Verizon iPhone customers complained their...   
9092  Ï¡Ïàü_ÊÎÒ£Áââ_£â_ÛâRT @...   

     emotion_in_tweet_is_directed_at  \
0                             iPhone   
1                 iPad or iPhone App   
2                               iPad   
3                 iPad or iPhone App   
4                             Google   
...                

In [29]:
raw_df.describe

<bound method NDFrame.describe of                                              tweet_text  \
0     .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1     @jessedee Know about @fludapp ? Awesome iPad/i...   
2     @swonderlin Can not wait for #iPad 2 also. The...   
3     @sxsw I hope this year's festival isn't as cra...   
4     @sxtxstate great stuff on Fri #SXSW: Marissa M...   
...                                                 ...   
9088                      Ipad everywhere. #SXSW {link}   
9089  Wave, buzz... RT @mention We interrupt your re...   
9090  Google's Zeiger, a physician never reported po...   
9091  Some Verizon iPhone customers complained their...   
9092  Ï¡Ïàü_ÊÎÒ£Áââ_£â_ÛâRT @...   

     emotion_in_tweet_is_directed_at  \
0                             iPhone   
1                 iPad or iPhone App   
2                               iPad   
3                 iPad or iPhone App   
4                             Google   
...              

In [30]:
raw_df.shape

(9093, 3)

In [31]:
raw_df.value_counts

<bound method DataFrame.value_counts of                                              tweet_text  \
0     .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1     @jessedee Know about @fludapp ? Awesome iPad/i...   
2     @swonderlin Can not wait for #iPad 2 also. The...   
3     @sxsw I hope this year's festival isn't as cra...   
4     @sxtxstate great stuff on Fri #SXSW: Marissa M...   
...                                                 ...   
9088                      Ipad everywhere. #SXSW {link}   
9089  Wave, buzz... RT @mention We interrupt your re...   
9090  Google's Zeiger, a physician never reported po...   
9091  Some Verizon iPhone customers complained their...   
9092  Ï¡Ïàü_ÊÎÒ£Áââ_£â_ÛâRT @...   

     emotion_in_tweet_is_directed_at  \
0                             iPhone   
1                 iPad or iPhone App   
2                               iPad   
3                 iPad or iPhone App   
4                             Google   
...        

3.2. Data Cleaning

In [32]:
#renaming columns for cleaner coding access
raw_df.columns = ['tweet_text', 'product_target', 'sentiment']

In [33]:
#handle missing values
raw_df.isna().value_counts()#checking the missing values

tweet_text  product_target  sentiment
False       True            False        5801
            False           False        3291
True        True            False           1
Name: count, dtype: int64

In [34]:
# Target: 'sentiment' contains no missing values, but 'tweet_text' has 1 null row.
#drop the missing values
cleaned_df = raw_df.dropna(subset=['tweet_text']).copy()

In [35]:
#Filter Out Target Ambiguity 
# Remove human annotator noise ("I can't tell") to preserve definitive sentiment vectors
cleaned_df = cleaned_df[cleaned_df['sentiment'] != "I can't tell"]

In [36]:
#checking for duplicates
duplicate_count = cleaned_df.duplicated(subset=['tweet_text']).sum()
duplicate_count

27

In [37]:
#drop the duplicates
cleaned_df = cleaned_df.drop_duplicates(subset=['tweet_text'], keep='first')

In [38]:
#Target numerical encoding
#cast text classes into machine-readable format
target_encoder = {
    'Negative emotion': 0,
    'No emotion toward brand or product': 1,
    'Positive emotion': 2
}
cleaned_df['label'] = cleaned_df['sentiment'].map(target_encoder)

print(f"Final post-cleaned dataframe structural dimensions: {cleaned_df.shape}\n")

Final post-cleaned dataframe structural dimensions: (8909, 4)



In [ ]:
#check for multicollinearity
print("--- MULTICOLLINEARITY ANALYSIS VIA VARIANCE INFLATION FACTOR (VIF) ---")

--- MULTICOLLINEARITY ANALYSIS VIA VARIANCE INFLATION FACTOR (VIF) ---


In [46]:
#feature engineer structural text metadata features to test numerical stability
cleaned_df['char_length'] = cleaned_df['tweet_text'].apply(len)
cleaned_df['word_count'] = cleaned_df['tweet_text'].apply(lambda x: len(x.split()))
# Add a minor smoothing term (1e-5) to prevent accidental division by zero on empty strings
cleaned_df['avg_word_length'] = cleaned_df['char_length'] / (cleaned_df['word_count'] + 1e-5)

# Isolate the constructed numeric features matrix
numerical_features = ['char_length', 'word_count', 'avg_word_length']
vif_matrix = cleaned_df[numerical_features].dropna()

# Compute standalone VIF coefficients
vif_data = pd.DataFrame()
vif_data["Feature"] = vif_matrix.columns
vif_data["VIF_Score"] = [
    variance_inflation_factor(vif_matrix.values, i) 
    for i in range(len(vif_matrix.columns))
]

# Display scores rounded to two decimal places
print(vif_data.round(2))

           Feature  VIF_Score
0      char_length     124.25
1       word_count      81.87
2  avg_word_length      16.19


Interpretation of Multicollinearity & Feature Strategy

*The Problem (High Multicollinearity):* When we examine basic text traits like character count and word count, they are highly dependent on each other. A tweet with more words will always have more characters. This mathematical redundancy is called multicollinearity, and it is flagged by a Variance Inflation Factor (VIF) score greater than $5.0$.
* **Why It Matters:** If we feed highly redundant, overlapping count metrics into a machine learning model, the model becomes unstable and confused. It cannot accurately determine which specific feature is actually driving the sentiment prediction.
* **The Solution (Our Feature Strategy):** To solve this, we bypass absolute count numbers entirely. Instead, we convert the raw text using a **TF-IDF Vectorizer**. TF-IDF focuses on the relative frequency and uniqueness of words across the dataset, which naturally eliminates numerical redundancy and provides our models with clean, independent feature